# Balanced Dataset Sampling dari output.jsonl

Notebook ini mengambil **1000 sample** dari `output.jsonl` secara **seimbang antar desil** (100 record per desil, dari Desil 1 sampai Desil 10).

Tujuan: Mengurangi ukuran dataset dari ~100.000 record menjadi 1.000 record untuk proses fine-tuning yang lebih efisien secara komputasi.

---
**Struktur output:**
- Format: JSONL (ChatML)
- Total sample: 1000
- Per desil: 100 record
- Output: `data/processed/output_sampled_1k.jsonl`

In [ ]:
import json
import re
import random
from collections import defaultdict
from pathlib import Path

# Seed untuk reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print('Libraries loaded ✓')

In [ ]:
# ============================================================
# Konfigurasi path
# ============================================================
BASE_DIR = Path('.')  # root dari data_pipeline/
INPUT_FILE  = BASE_DIR / 'data' / 'processed' / 'output.jsonl'
OUTPUT_FILE = BASE_DIR / 'data' / 'processed' / 'output_sampled_1k.jsonl'

TOTAL_SAMPLES   = 1000   # total sample yang diinginkan
NUM_DECILES     = 10     # Desil 1 – Desil 10
SAMPLES_PER_DECILE = TOTAL_SAMPLES // NUM_DECILES  # 100 per desil

print(f'Input  : {INPUT_FILE}')
print(f'Output : {OUTPUT_FILE}')
print(f'Target : {SAMPLES_PER_DECILE} samples × {NUM_DECILES} desil = {TOTAL_SAMPLES} total')

In [ ]:
# ============================================================
# Fungsi helper: ekstrak nomor desil dari field assistant
# ============================================================
# Format assistant baru: '...\nDesil Nasional: 7'
DECILE_PATTERN = re.compile(r'Desil Nasional:\s*(\d+)', re.IGNORECASE)

def extract_decile(record: dict) -> int | None:
    """Ekstrak nomor desil (1-10) dari pesan assistant."""
    try:
        messages = record.get('messages', [])
        for msg in messages:
            if msg.get('role') == 'assistant':
                match = DECILE_PATTERN.search(msg['content'])
                if match:
                    return int(match.group(1))
    except Exception:
        pass
    return None

# Uji fungsi pada contoh format assistant baru
sample_record = {
    'messages': [
        {'role': 'assistant', 'content': 'Analisis Kondisi:\n- Demografi & Lokasi: ...\n\nReasoning: ...\n\nSkor Evaluasi: 0.75\nDesil Nasional: 8'}
    ]
}
assert extract_decile(sample_record) == 8, 'Fungsi extract_decile gagal!'
print('Fungsi extract_decile OK ✓')

In [ ]:
# ============================================================
# Baca & kelompokkan semua record berdasarkan desil
# ============================================================
decile_buckets: dict[int, list] = defaultdict(list)
skipped = 0

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            skipped += 1
            continue

        decile = extract_decile(record)
        if decile is None or not (1 <= decile <= 10):
            skipped += 1
            continue

        decile_buckets[decile].append(record)

print(f'Total baris dibaca   : {line_num:,}')
print(f'Baris dilewati (err) : {skipped:,}')
print()
for d in sorted(decile_buckets):
    print(f'  Desil {d:>2}: {len(decile_buckets[d]):>8,} records')

In [ ]:
# ============================================================
# Validasi: pastikan setiap desil memiliki cukup data
# ============================================================
insufficient = [
    d for d in range(1, NUM_DECILES + 1)
    if len(decile_buckets[d]) < SAMPLES_PER_DECILE
]

if insufficient:
    print(f'⚠️  PERINGATAN: Desil berikut memiliki data < {SAMPLES_PER_DECILE}:')
    for d in insufficient:
        print(f'   Desil {d}: hanya {len(decile_buckets[d])} records (akan diambil semua)')
else:
    print(f'✓ Semua desil memiliki ≥ {SAMPLES_PER_DECILE} records. Sampling 100 per desil.')

In [ ]:
# ============================================================
# Lakukan random sampling seimbang per desil
# ============================================================
sampled_records = []

for decile in range(1, NUM_DECILES + 1):
    bucket = decile_buckets[decile]
    n = min(SAMPLES_PER_DECILE, len(bucket))  # fallback jika bucket < 100
    sampled = random.sample(bucket, k=n)
    sampled_records.extend(sampled)
    print(f'  Desil {decile:>2}: ambil {n:>3} dari {len(bucket):>8,} records')

# Acak urutan final agar tidak ordered by desil
random.shuffle(sampled_records)

print(f'\nTotal sample terkumpul: {len(sampled_records):,}')

In [ ]:
# ============================================================
# Tulis hasil ke file JSONL baru
# ============================================================
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for record in sampled_records:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

output_size_mb = OUTPUT_FILE.stat().st_size / (1024 ** 2)

print(f'✓ Output tersimpan ke: {OUTPUT_FILE}')
print(f'  Jumlah baris : {len(sampled_records):,}')
print(f'  Ukuran file  : {output_size_mb:.2f} MB')

In [ ]:
# ============================================================
# Verifikasi distribusi desil pada output file
# ============================================================
verify_buckets = defaultdict(int)

with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line.strip())
        decile = extract_decile(record)
        if decile:
            verify_buckets[decile] += 1

print('Distribusi desil pada output_sampled_1k.jsonl:')
print('-' * 40)
total_verified = 0
for d in sorted(verify_buckets):
    count = verify_buckets[d]
    bar = '█' * (count // 2)
    print(f'  Desil {d:>2}: {count:>4} records  {bar}')
    total_verified += count
print('-' * 40)
print(f'  Total    : {total_verified:>4} records')
print()

# Cek keseimbangan
counts = list(verify_buckets.values())
if len(set(counts)) == 1:
    print(f'✅ Dataset SEIMBANG: setiap desil memiliki tepat {counts[0]} records.')
else:
    print(f'⚠️  Dataset tidak sepenuhnya seimbang. Min={min(counts)}, Max={max(counts)}')

In [ ]:
# ============================================================
# Preview: tampilkan 1 contoh per desil
# ============================================================
print('Preview 1 record per desil (hanya respons assistant):') 
print('=' * 70)

seen_deciles = set()
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line.strip())
        decile = extract_decile(record)
        if decile and decile not in seen_deciles:
            seen_deciles.add(decile)
            assistant_msg = next(
                (m['content'] for m in record['messages'] if m['role'] == 'assistant'),
                '(tidak ditemukan)'
            )
            print(f'\n[Desil {decile}]')
            print(f'  {assistant_msg}')
        if len(seen_deciles) == NUM_DECILES:
            break

print('\n' + '=' * 70)
print('✅ Sampling selesai! Dataset siap untuk fine-tuning.')